# ValCalib GPTQ Search / Rescue / Rotation

This notebook tests export-time search features on the March 25 record-style Colab stack.

Compare runs by the final log line `final_int6_sliding_window_s64_exact`.

The run cells use `GPTQ_AR_CALIB_NUM_SEQS=8` and `GPTQ_AR_CALIB_SEQ_LEN=512` for Colab smoke speed. Remove those overrides for a fuller record-aligned export pass.

Recommended order: baseline parity, search infra only, mixed precision only, rotation only, combined search.

In [ ]:
!git clone https://github.com/IanniMuliterno/parameter-golf.git || true
%cd /content/parameter-golf/colab/2026-04-12_ValCalib_GPTQ_SearchRescueRotation
!python3 -m pip install -r requirements.txt

## 1. Baseline parity

Search disabled. This should behave like the March 25 Colab replica, but from this patched folder.

In [ ]:
!RUN_ID=baseline_parity \
ITERATIONS=300 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=1 \
GPTQ_AR_CALIB_NUM_SEQS=8 \
GPTQ_AR_CALIB_SEQ_LEN=512 \
GPTQ_AR_CALIB_BATCH_SIZE=4 \
EXPORT_SEARCH_ENABLED=0 \
bash run.sh

## 2. Search infrastructure only

Same export settings as baseline, but routed through the candidate/frontier path.

In [ ]:
!RUN_ID=search_infra_only \
ITERATIONS=300 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=1 \
GPTQ_AR_CALIB_NUM_SEQS=8 \
GPTQ_AR_CALIB_SEQ_LEN=512 \
GPTQ_AR_CALIB_BATCH_SIZE=4 \
EXPORT_SEARCH_ENABLED=1 \
SEARCH_ROTATION_OPTIONS=0 \
SEARCH_MIXED_PRECISION_BUDGETS=0 \
SEARCH_MIXED_PRECISION_MAX_TENSORS=0 \
SEARCH_GPTQ_BLOCK_SIZES=128 \
bash run.sh

## 3. Mixed precision only

Searches a small fp16 rescue budget for sensitive large tensors. Rotation remains disabled.

In [ ]:
!RUN_ID=mixed_precision_only \
ITERATIONS=300 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=1 \
GPTQ_AR_CALIB_NUM_SEQS=8 \
GPTQ_AR_CALIB_SEQ_LEN=512 \
GPTQ_AR_CALIB_BATCH_SIZE=4 \
EXPORT_SEARCH_ENABLED=1 \
SEARCH_ROTATION_OPTIONS=0 \
SEARCH_MIXED_PRECISION_BUDGETS=0,65536,131072,262144 \
SEARCH_MIXED_PRECISION_MAX_TENSORS=0,1,2 \
SEARCH_GPTQ_BLOCK_SIZES=128 \
bash run.sh

## 4. Rotation only

Searches rotation off/on and GPTQ block size. Mixed precision rescue remains disabled.

In [ ]:
!RUN_ID=rotation_only \
ITERATIONS=300 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=1 \
GPTQ_AR_CALIB_NUM_SEQS=8 \
GPTQ_AR_CALIB_SEQ_LEN=512 \
GPTQ_AR_CALIB_BATCH_SIZE=4 \
EXPORT_SEARCH_ENABLED=1 \
ROTATION_AWARE_ENABLED=1 \
SEARCH_ROTATION_OPTIONS=0,1 \
SEARCH_MIXED_PRECISION_BUDGETS=0 \
SEARCH_MIXED_PRECISION_MAX_TENSORS=0 \
SEARCH_GPTQ_BLOCK_SIZES=64,128 \
HESSIAN_DAMPING=0.03 \
GPTQ_ACCEPT_MSE_RATIO=1.03 \
bash run.sh

## 5. Mixed precision plus rotation

This is the full search pass. It is slower because frontier candidates are roundtrip-evaluated.

In [ ]:
!RUN_ID=mixed_precision_plus_rotation \
ITERATIONS=300 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=1 \
GPTQ_AR_CALIB_NUM_SEQS=8 \
GPTQ_AR_CALIB_SEQ_LEN=512 \
GPTQ_AR_CALIB_BATCH_SIZE=4 \
EXPORT_SEARCH_ENABLED=1 \
ROTATION_AWARE_ENABLED=1 \
SEARCH_ROTATION_OPTIONS=0,1 \
SEARCH_MIXED_PRECISION_BUDGETS=0,65536,131072,262144 \
SEARCH_MIXED_PRECISION_MAX_TENSORS=0,1,2 \
SEARCH_GPTQ_BLOCK_SIZES=64,128 \
HESSIAN_DAMPING=0.03 \
GPTQ_ACCEPT_MSE_RATIO=1.03 \
SEARCH_MAX_FRONTIER_EVALS=6 \
bash run.sh

## Inspect tables

Run after any search-enabled cell.

In [ ]:
!ls -lh logs || true
!tail -n 40 logs/*.txt || true
!python3 - <<'PY'
from pathlib import Path
for name in ['artifact_size_table.tsv', 'search_frontier.tsv', 'roundtrip_quality_table.tsv']:
    path = Path('logs') / name
    print(f'\n=== {name} ===')
    if path.exists():
        print(path.read_text()[:4000])
    else:
        print('missing')
PY